## Cell 1 — Setup & config

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, re, json, time, unicodedata, getpass, warnings
from datetime import date
from difflib import SequenceMatcher

import numpy as np
import pandas as pd
import requests
import networkx as nx

!pip install -q ftfy scikit-learn scipy
import ftfy
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.spatial.distance import squareform
from networkx.algorithms.community import louvain_communities, modularity as nx_modularity

warnings.filterwarnings('ignore')

OUT_DIR = '/content/drive/MyDrive/openalex_enrich_out'
PAPERS_IN = f'{OUT_DIR}/nips-papers_enriched_openalex.csv'
PAPERS_OUT = f'{OUT_DIR}/nips-papers-fixed.csv'
C2_CKPT = f'{OUT_DIR}/nips-c2-checkpoint.csv'
C2_OUT = f'{OUT_DIR}/nips-papers-c2.csv'
TOPIC_MAP_OUT = f'{OUT_DIR}/nips-topic-merge-map.json'
PANEL_OUT = f'{OUT_DIR}/nips-panel-v4.csv'
VARDESC_OUT = f'{OUT_DIR}/nips-variable-descriptions.json'
EXPORT_DIR = f'{OUT_DIR}/dag_learning_ready'

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(EXPORT_DIR, exist_ok=True)

COMPLETE = {'ok', 'ok_fallback', 'ok_manual_doi', 'ok_refetch'}
EXCLUDE_SRC = {4581, 5987, 5346, 14379, 15556, 18507, 20437}
MIN_CELL_SIZE = 20
MIN_YEAR_SPAN = 7
EXCLUDE_YEAR = 2024
MERGE_THRESHOLD = 0.50
CURRENT_YEAR = date.today().year
SNAPSHOT = str(date.today())

print(f'{SNAPSHOT} | min_cell={MIN_CELL_SIZE} | min_years={MIN_YEAR_SPAN}')

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.7 MB/s eta 0:00:00
2026-03-17 | min_cell=20 | min_years=7


## Cell 2 — API key & shared helpers

In [ ]:
def clean_title(x):
    if not isinstance(x, str):
        return x
    x = ftfy.fix_text(unicodedata.normalize('NFC', x))
    x = re.sub(r'\$.*?\$|\\[a-zA-Z]+|[{}]', ' ', x)
    return re.sub(r'\s+', ' ', x).strip()

def title_sim(a, b):
    a = re.sub(r'\s+', ' ', str(a).strip().lower())
    b = re.sub(r'\s+', ' ', str(b).strip().lower())
    return SequenceMatcher(None, a, b).ratio()

def parse_json_list(x):
    if isinstance(x, list):
        return x
    if pd.isna(x) or x in ('', '[]', None):
        return []
    try:
        x = json.loads(x)
    except Exception:
        return []
    return x if isinstance(x, list) else []

def work_to_fields(work):
    pt = work.get('primary_topic') or {}
    auths = work.get('authorships') or []
    ids = [a['author']['id'] for a in auths if a.get('author', {}).get('id')]
    names = [a['author'].get('display_name') for a in auths if a.get('author', {}).get('id')]
    pos = [a.get('author_position') for a in auths if a.get('author', {}).get('id')]
    topics = [{'id': t.get('id'), 'name': t.get('display_name'), 'score': t.get('score')} for t in (work.get('topics') or [])[:5]]
    return {
        'oa_work_id': work.get('id'),
        'oa_display_name': work.get('display_name'),
        'oa_doi': (work.get('doi') or '').replace('https://doi.org/', ''),
        'oa_cited_by_count': work.get('cited_by_count'),
        'oa_primary_topic': pt.get('display_name'),
        'oa_domain': (pt.get('domain') or {}).get('display_name'),
        'oa_field': (pt.get('field') or {}).get('display_name'),
        'oa_subfield': (pt.get('subfield') or {}).get('display_name'),
        'oa_topics_top5_json': json.dumps(topics, ensure_ascii=False),
        'oa_author_ids_json': json.dumps(ids, ensure_ascii=False),
        'oa_author_names_json': json.dumps(names, ensure_ascii=False),
        'oa_author_positions_json': json.dumps(pos, ensure_ascii=False),
    }

OA_KEY = getpass.getpass('OpenAlex API key: ').strip()
assert OA_KEY
S = requests.Session()
SELECT_SLIM = 'id,display_name,publication_year,doi,cited_by_count,primary_topic,topics,authorships'

def oa_search(title, year, k=5, sim_min=0.85):
    q = '"' + title.replace('\\', '\\\\').replace('"', '\\"') + '"'
    for filt in [f'title.search:{q},publication_year:{year-1}|{year}|{year+1}', f'title.search:{q}']:
        r = S.get('https://api.openalex.org/works', params={'filter': filt, 'per_page': k, 'select': SELECT_SLIM, 'api_key': OA_KEY}, timeout=30)
        if r.status_code == 429:
            raise RuntimeError('Rate limited')
        if r.status_code != 200:
            continue
        res = r.json().get('results') or []
        if not res:
            continue
        best = max(res, key=lambda w: title_sim(title, w.get('display_name', '')))
        sim = title_sim(title, best.get('display_name', ''))
        by = best.get('publication_year')
        if sim >= sim_min and (by is None or abs(int(by) - year) <= 2):
            return best, 'ok'
    return None, 'no_match'

OpenAlex API key: ··········


## Cell 3 — Fix loop *(re-run until ghost_rows = 0)*

In [ ]:
papers = pd.read_csv(PAPERS_OUT if os.path.exists(PAPERS_OUT) else PAPERS_IN)

m = papers['src_index'].astype(int) == 7049
if m.any() and str(papers.loc[m, 'oa_work_id'].iloc[0]) != 'https://openalex.org/W2626778328':
    r = S.get('https://api.openalex.org/works/W2626778328', params={'select': SELECT_SLIM, 'api_key': OA_KEY}, timeout=30)
    if r.status_code == 429:
        raise RuntimeError('Rate limited')
    vals = {'oa_status': 'ok_manual_doi', 'oa_title_sim': 1.0, **work_to_fields(r.json())}
    for c, v in vals.items():
        if c in papers.columns:
            papers.loc[m, c] = v

ghost = papers['oa_status'].isin(COMPLETE) & papers['oa_work_id'].isna()
todo = papers.index[ghost].tolist()

for i, idx in enumerate(todo, 1):
    row = papers.loc[idx]
    title = clean_title(row.get('title', ''))
    if not title or len(title) < 5:
        papers.loc[idx, 'oa_status'] = 'no_match'
        continue
    work, status = oa_search(title, int(row['year']))
    if work is None:
        papers.loc[idx, 'oa_status'] = status
        continue
    vals = {'oa_status': 'ok_refetch', 'oa_title_sim': title_sim(title, work.get('display_name', '')), **work_to_fields(work)}
    for c, v in vals.items():
        if c in papers.columns:
            papers.loc[idx, c] = v
    if i % 50 == 0:
        papers.to_csv(PAPERS_OUT, index=False)
        print(i, len(todo))
    time.sleep(0.15)

ghost = papers['oa_status'].isin(COMPLETE) & papers['oa_work_id'].isna()
if ghost.any():
    papers.loc[ghost, 'oa_status'] = 'missing_in_openalex'

papers['snapshot_date'] = SNAPSHOT
papers.to_csv(PAPERS_OUT, index=False)
print(int((papers['oa_status'].isin(COMPLETE) & papers['oa_work_id'].isna()).sum()))

0


## Cell 4 — Windowed citations *(re-run until remaining = 0)*

Computes **C2** (pub_year + pub_year+1) as the primary citation window.
C3 stored alongside as sensitivity.

In [ ]:
papers = pd.read_csv(PAPERS_OUT)

matched = papers[papers['oa_status'].isin(COMPLETE) & papers['oa_work_id'].notna() & (papers['year'].astype(int) != EXCLUDE_YEAR)].copy()
matched['_uri'] = matched['oa_work_id'].str.strip()

ckpt = pd.read_csv(C2_CKPT) if os.path.exists(C2_CKPT) else pd.DataFrame(columns=['oa_work_id', 'counts_by_year_json'])
done = set(ckpt['oa_work_id'].dropna())
todo = [u for u in matched['_uri'] if u not in done]

rows = []
for b, start in enumerate(range(0, len(todo), 50), 1):
    batch = todo[start:start + 50]
    r = S.get('https://api.openalex.org/works', params={'filter': f'ids.openalex:{"|".join(batch)}', 'per_page': len(batch), 'select': 'id,counts_by_year', 'api_key': OA_KEY}, timeout=45)
    if r.status_code == 429:
        raise RuntimeError('Rate limited')
    if r.status_code != 200:
        continue
    got = {x['id']: x.get('counts_by_year') for x in (r.json().get('results') or [])}
    rows.extend({'oa_work_id': u, 'counts_by_year_json': json.dumps(got.get(u), ensure_ascii=False) if got.get(u) is not None else None} for u in batch)
    if b % 50 == 0:
        ckpt = pd.concat([ckpt, pd.DataFrame(rows)], ignore_index=True).drop_duplicates('oa_work_id', keep='last')
        ckpt.to_csv(C2_CKPT, index=False)
        rows = []
        print(b)
    time.sleep(0.1)

if rows:
    ckpt = pd.concat([ckpt, pd.DataFrame(rows)], ignore_index=True)
ckpt = ckpt.drop_duplicates('oa_work_id', keep='last')
ckpt.to_csv(C2_CKPT, index=False)

def compute_c2_c3(cby_json, pub_year):
    if pd.isna(cby_json):
        return pd.Series([np.nan, np.nan, False], index=['c2', 'c3', 'c2_complete'])
    try:
        counts = {x['year']: x['cited_by_count'] for x in json.loads(cby_json) if isinstance(x, dict)}
    except Exception:
        return pd.Series([np.nan, np.nan, False], index=['c2', 'c3', 'c2_complete'])
    c2 = counts.get(pub_year, 0) + counts.get(pub_year + 1, 0)
    c3 = c2 + counts.get(pub_year + 2, 0) if CURRENT_YEAR > pub_year + 2 else np.nan
    return pd.Series([c2, c3, CURRENT_YEAR > pub_year + 1], index=['c2', 'c3', 'c2_complete'])

matched['_cby'] = matched['_uri'].map(ckpt.drop_duplicates('oa_work_id').set_index('oa_work_id')['counts_by_year_json'])
matched[['c2', 'c3', 'c2_complete']] = matched.apply(lambda r: compute_c2_c3(r['_cby'], int(r['year'])), axis=1)

papers = papers.merge(matched[['src_index', 'c2', 'c3', 'c2_complete']], on='src_index', how='left')
papers.to_csv(C2_OUT, index=False)
print(int(papers['c2'].notna().sum()))

15636


## Cell 5 — Automated topic merging

In [ ]:
papers = pd.read_csv(C2_OUT)
df = papers[papers['oa_status'].isin(COMPLETE) & papers['oa_work_id'].notna() & (papers['year'].astype(int) != EXCLUDE_YEAR) & (~papers['src_index'].isin(EXCLUDE_SRC)) & papers['oa_primary_topic'].notna()].copy()
topics = sorted(df['oa_primary_topic'].str.strip().unique())

tfidf = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5)).fit_transform(topics)
dist = np.clip(1 - cosine_similarity(tfidf), 0, None)
np.fill_diagonal(dist, 0)
labels = fcluster(linkage(squareform(dist, checks=False), method='average'), t=1 - MERGE_THRESHOLD, criterion='distance')

counts = df['oa_primary_topic'].str.strip().value_counts()
clusters = {}
for topic, label in zip(topics, labels):
    clusters.setdefault(label, []).append(topic)

merge_map = {}
for members in clusters.values():
    if len(members) < 2:
        continue
    canon = max(members, key=lambda x: counts.get(x, 0))
    merge_map.update({x: canon for x in members if x != canon})

with open(TOPIC_MAP_OUT, 'w') as f:
    json.dump(merge_map, f, indent=2)

df['topic'] = df['oa_primary_topic'].str.strip().map(merge_map).fillna(df['oa_primary_topic'].str.strip())
print(df['topic'].nunique())


548


## Cell 6 — Build panel

| Role | Variable | Definition |
|------|----------|------------|
| Predictor (t−1) | TopicShare | papers in topic k / all NeurIPS papers |
| Predictor (t−1) | CrossTopicAuthorRate | % authors in ≥2 topics |
| Predictor (t−1) | Connectivity | mean degree of co-author graph |
| Predictor (t−1) | Modularity | Louvain modularity of co-author graph |
| Predictor (t−1) | BridgeConcentration | betweenness share held by top 10% authors |
| Outcome (t) | TopicGrowth | log-ratio of TopicShare |
| Outcome (t) | MedianCites2yr | median per-paper C2 |
| Outcome (t) | HitRate2yr | % papers in top 10% of C2 |


In [ ]:
papers = pd.read_csv(C2_OUT)
merge_map = json.load(open(TOPIC_MAP_OUT)) if os.path.exists(TOPIC_MAP_OUT) else {}

df = papers[papers['oa_status'].isin(COMPLETE) & papers['oa_work_id'].notna() & (papers['year'].astype(int) != EXCLUDE_YEAR) & (~papers['src_index'].isin(EXCLUDE_SRC)) & papers['oa_primary_topic'].notna()].copy()
df['year'] = df['year'].astype(int)
df['c2'] = pd.to_numeric(df.get('c2'), errors='coerce')
df['c3'] = pd.to_numeric(df.get('c3'), errors='coerce')
df['topic'] = df['oa_primary_topic'].str.strip().map(merge_map).fillna(df['oa_primary_topic'].str.strip())
df['_authors'] = df['oa_author_ids_json'].apply(parse_json_list)
df['_topics_top5'] = df['oa_topics_top5_json'].apply(parse_json_list)
df['_team_size'] = df['_authors'].str.len()

topic_df = df[df['topic'].notna()].copy()
yearly_total = df.groupby('year').size()
c2_base = df[df['c2_complete'].fillna(False)] if 'c2_complete' in df else df.iloc[0:0]
yr_p90 = c2_base.groupby('year')['c2'].quantile(0.90) if len(c2_base) else pd.Series(dtype=float)

at = pd.DataFrame(
    [{'aid': aid, 'year': y, 'topic': t} for y, t, ids in topic_df[['year', 'topic', '_authors']].itertuples(index=False) for aid in ids if isinstance(aid, str)],
    columns=['aid', 'year', 'topic']
).drop_duplicates()
cross = at.groupby(['aid', 'year'])['topic'].nunique().ge(2).rename('cross').reset_index()
at = at.merge(cross, on=['aid', 'year'], how='left')

def make_graph(author_lists):
    G = nx.Graph()
    for ids in author_lists:
        ids = [a for a in ids if isinstance(a, str)]
        G.add_nodes_from(ids)
        for i, a in enumerate(ids):
            for b in ids[i + 1:]:
                if G.has_edge(a, b):
                    G[a][b]['weight'] += 1
                else:
                    G.add_edge(a, b, weight=1)
    return G

def graph_metrics(author_lists):
    G = make_graph(author_lists)
    if len(G) < 3 or G.number_of_edges() == 0:
        return len(G), np.nan, np.nan, np.nan
    deg = float(np.mean([d for _, d in G.degree()]))
    comms = louvain_communities(G, weight='weight', seed=42)
    mod = nx_modularity(G, comms, weight='weight')
    bc = sorted(nx.betweenness_centrality(G, weight='weight').values(), reverse=True)
    k = max(1, int(len(bc) * 0.10))
    bc_share = sum(bc[:k]) / sum(bc) if sum(bc) else 0.0
    return len(G), deg, mod, bc_share

def gini(x):
    x = np.sort(np.asarray(x, float))
    n = len(x)
    return np.nan if n == 0 or x.sum() == 0 else (2 * np.sum(np.arange(1, n + 1) * x) - (n + 1) * x.sum()) / (n * x.sum())

rows = []
for (year, topic), cell in topic_df.groupby(['year', 'topic'], sort=True):
    n_auth = at[(at['year'] == year) & (at['topic'] == topic)]['aid'].nunique()
    cross_r = at[(at['year'] == year) & (at['topic'] == topic)].drop_duplicates('aid')['cross'].mean()
    n_nodes, mean_deg, mod, bc_share = graph_metrics(cell['_authors'])
    cell_c = cell[cell['c2_complete'].fillna(False)] if 'c2_complete' in cell else cell.iloc[0:0]
    c2v = cell_c['c2'].dropna().to_numpy()
    c3v = cell_c['c3'].dropna().to_numpy()
    p90 = yr_p90.get(year, np.nan)
    rows.append({
        'topic': topic,
        'year': year,
        'n_papers': len(cell),
        'n_authors': n_nodes,
        'topic_share': len(cell) / yearly_total.get(year, 1),
        'cross_topic_rate': float(cross_r) if pd.notna(cross_r) else np.nan,
        'connectivity': mean_deg,
        'modularity': mod,
        'bridge_concentration': bc_share,
        'median_cites_2yr': float(np.median(c2v)) if len(c2v) else np.nan,
        'hit_rate_2yr': float((c2v >= p90).mean()) if len(c2v) and pd.notna(p90) else np.nan,
        'avg_team_size': float(cell['_team_size'].mean()),
        'median_cites_3yr': float(np.median(c3v)) if len(c3v) else np.nan,
        'citation_gini': gini(c2v) if len(c2v) >= 3 else np.nan,
    })

panel = pd.DataFrame(rows).sort_values(['topic', 'year']).reset_index(drop=True)
panel['topic_growth'] = np.log(panel['topic_share'] + 1e-4) - np.log(panel.groupby('topic')['topic_share'].shift(1) + 1e-4)
panel['log1p_median_c2'] = np.log1p(panel['median_cites_2yr'])

lag_cols = ['topic_share', 'cross_topic_rate', 'connectivity', 'modularity', 'bridge_concentration']
lag = panel[['topic', 'year'] + lag_cols].rename(columns={c: f'{c}_t1' for c in lag_cols})
lag['year'] += 1
panel = panel.merge(lag, on=['topic', 'year'], how='inner')

keep_topics = panel.groupby('topic')['year'].nunique().ge(MIN_YEAR_SPAN)
panel = panel[panel['topic'].isin(keep_topics[keep_topics].index) & panel['n_papers'].ge(MIN_CELL_SIZE)].copy()

out_cols = [
    'topic', 'year', 'n_papers', 'n_authors',
    'topic_share_t1', 'cross_topic_rate_t1', 'connectivity_t1', 'modularity_t1', 'bridge_concentration_t1',
    'topic_growth', 'median_cites_2yr', 'log1p_median_c2', 'hit_rate_2yr',
    'topic_share', 'cross_topic_rate', 'connectivity', 'modularity', 'bridge_concentration',
    'avg_team_size', 'median_cites_3yr', 'citation_gini',
]
out = panel[[c for c in out_cols if c in panel.columns]].sort_values(['topic', 'year']).reset_index(drop=True)
out.to_csv(PANEL_OUT, index=False)

core = ['topic_share_t1', 'cross_topic_rate_t1', 'connectivity_t1', 'modularity_t1', 'bridge_concentration_t1', 'topic_growth', 'median_cites_2yr', 'hit_rate_2yr']
print(out.shape, int(out[core].notna().all(axis=1).sum()))


(156, 21) 156


## Cell 7 — Variable descriptions

In [ ]:
variables = [
    ('topic_share_t1', 'Share of NeurIPS papers in topic k at t-1', 'bounded proportion', 'predictor'),
    ('cross_topic_rate_t1', 'Share of topic-k authors at t-1 who also publish in another topic', 'bounded proportion', 'predictor'),
    ('connectivity_t1', 'Mean degree of the topic-k coauthorship graph at t-1', 'continuous', 'predictor'),
    ('modularity_t1', 'Louvain modularity of the topic-k coauthorship graph at t-1', 'continuous', 'predictor'),
    ('bridge_concentration_t1', 'Share of total betweenness held by the top 10% of authors at t-1', 'bounded proportion', 'predictor'),
    ('topic_growth', 'log(topic_share_t + eps) - log(topic_share_t-1 + eps)', 'continuous', 'outcome'),
    ('median_cites_2yr', 'Median 2-year citations for topic-k papers at t', 'count', 'outcome'),
    ('hit_rate_2yr', 'Share of topic-k papers at t above the year-level 90th percentile in C2', 'bounded proportion', 'outcome'),
]

output = {
    'variables': [{'name': n, 'short_description': d, 'type': t, 'temporal_role': r, 'direction_constraints': 'no t -> t-1 edges'} for n, d, t, r in variables],
    'domain_context': {
        'dataset': 'NeurIPS proceedings 2012-2023',
        'unit_of_analysis': 'row = (topic, year)',
        'causal_question': 'Does topic-level collaboration structure at t-1 predict topic growth and early citation impact at t?',
        'temporal_constraint': 't outcomes cannot point to t-1 predictors',
        'snapshot_date': SNAPSHOT,
        'min_papers_per_cell': MIN_CELL_SIZE,
        'citation_window': 'C2 primary, C3 sensitivity',
    },
}
with open(VARDESC_OUT, 'w') as f:
    json.dump(output, f, indent=2)
print(len(output['variables']))


8


## Cell 8 — Export for DAG learning

In [ ]:
import shutil

panel = pd.read_csv(PANEL_OUT)
core_cols = ['topic', 'year', 'topic_share_t1', 'cross_topic_rate_t1', 'connectivity_t1', 'modularity_t1', 'bridge_concentration_t1', 'topic_growth', 'median_cites_2yr', 'hit_rate_2yr']
core_panel = panel[[c for c in core_cols if c in panel.columns]].copy()
core_panel.to_csv(f'{EXPORT_DIR}/panel_core_8vars.csv', index=False)
shutil.copy2(PANEL_OUT, f'{EXPORT_DIR}/panel_full.csv')
shutil.copy2(VARDESC_OUT, f'{EXPORT_DIR}/variable_descriptions.json')

t1 = [c for c in core_cols if c.endswith('_t1')]
t = ['topic_growth', 'median_cites_2yr', 'hit_rate_2yr']
constraints = {
    'forbidden_edges': [[y, x] for y in t for x in t1],
    'temporal_tiers': {'tier_0_predictors_t1': t1, 'tier_1_outcomes_t': t},
}
with open(f'{EXPORT_DIR}/dag_constraints.json', 'w') as f:
    json.dump(constraints, f, indent=2)

complete = core_panel.drop(columns=['topic', 'year']).notna().all(axis=1).sum()
print(EXPORT_DIR, core_panel.shape, int(complete))


/content/drive/MyDrive/openalex_enrich_out/dag_learning_ready (156, 10) 156
